In [1]:
import pandas as pd

df = pd.read_excel("Masked_Kraft Heinz Securities Litigation.xlsx")

# Convert date
df["Trade Date"] = pd.to_datetime(df["Trade Date"], dayfirst=True)

In [2]:
df.head()

,,Fund Name,Fund Number,Transaction Type,Purchases,Sales,Holdings,Price per share,Trade Date,Currency,Entity,Comments
0,US5007541064,Fund 3,Fund 3,Beginning Holdings,NaN,NaN,38000.0,NaN,2015-11-05,USD,Client 1-1,NaN
1,US5007541064,Fund 3,Fund 3,Sale,NaN,2000.0,NaN,70.1700,2016-01-19,USD,Client 1-1,NaN
2,US5007541064,Fund 3,Fund 3,Purchase,1000.0,NaN,NaN,76.2900,2016-03-08,USD,Client 1-1,NaN
3,US5007541064,Fund 3,Fund 3,Purchase,1500.0,NaN,NaN,77.4000,2016-04-13,USD,Client 1-1,NaN
4,US5007541064,Fund 3,Fund 3,Sale,NaN,1500.0,NaN,90.3754,2016-08-11,USD,Client 1-1,NaN


In [3]:
def preprocess(df):
    df["Qty"] = df["Purchases"].fillna(0) - df["Sales"].fillna(0)

    df["Type"] = df["Transaction Type"].apply(lambda x: 
        "BUY" if "Purchase" in str(x) else 
        "SELL" if "Sale" in str(x) else 
        "OTHER"
    )

    return df

df = preprocess(df)

In [4]:
df.head()

,,Fund Name,Fund Number,Transaction Type,Purchases,Sales,Holdings,Price per share,Trade Date,Currency,Entity,Comments,Qty,Type
0,US5007541064,Fund 3,Fund 3,Beginning Holdings,NaN,NaN,38000.0,NaN,2015-11-05,USD,Client 1-1,NaN,0.0,OTHER
1,US5007541064,Fund 3,Fund 3,Sale,NaN,2000.0,NaN,70.1700,2016-01-19,USD,Client 1-1,NaN,-2000.0,SELL
2,US5007541064,Fund 3,Fund 3,Purchase,1000.0,NaN,NaN,76.2900,2016-03-08,USD,Client 1-1,NaN,1000.0,BUY
3,US5007541064,Fund 3,Fund 3,Purchase,1500.0,NaN,NaN,77.4000,2016-04-13,USD,Client 1-1,NaN,1500.0,BUY
4,US5007541064,Fund 3,Fund 3,Sale,NaN,1500.0,NaN,90.3754,2016-08-11,USD,Client 1-1,NaN,-1500.0,SELL


In [5]:
# Step 1: Clean relevant columns
df["Purchases"] = df["Purchases"].fillna(0)
df["Sales"] = df["Sales"].fillna(0)

# Step 2: Qty
df["Qty"] = df["Purchases"] - df["Sales"]

# Step 3: Filter valid rows
df = df[df["Transaction Type"].isin(["Purchase", "Sale"])]

In [6]:
df.head()

,,Fund Name,Fund Number,Transaction Type,Purchases,Sales,Holdings,Price per share,Trade Date,Currency,Entity,Comments,Qty,Type
1,US5007541064,Fund 3,Fund 3,Sale,0.0,2000.0,NaN,70.1700,2016-01-19,USD,Client 1-1,NaN,-2000.0,SELL
2,US5007541064,Fund 3,Fund 3,Purchase,1000.0,0.0,NaN,76.2900,2016-03-08,USD,Client 1-1,NaN,1000.0,BUY
3,US5007541064,Fund 3,Fund 3,Purchase,1500.0,0.0,NaN,77.4000,2016-04-13,USD,Client 1-1,NaN,1500.0,BUY
4,US5007541064,Fund 3,Fund 3,Sale,0.0,1500.0,NaN,90.3754,2016-08-11,USD,Client 1-1,NaN,-1500.0,SELL
5,US5007541064,Fund 3,Fund 3,Purchase,1000.0,0.0,NaN,87.2994,2017-02-16,USD,Client 1-1,NaN,1000.0,BUY


In [12]:
df = df[["Entity", "Fund Name", "Transaction Type", "Qty", "Price per share", "Trade Date"]]

In [13]:
df = df.sort_values(["Entity", "Fund Name", "Trade Date"])

In [14]:
df.head()

,Entity,Fund Name,Transaction Type,Qty,Price per share,Trade Date
1,Client 1-1,Fund 3,Sale,-2000.0,70.1700,2016-01-19
2,Client 1-1,Fund 3,Purchase,1000.0,76.2900,2016-03-08
3,Client 1-1,Fund 3,Purchase,1500.0,77.4000,2016-04-13
4,Client 1-1,Fund 3,Sale,-1500.0,90.3754,2016-08-11
5,Client 1-1,Fund 3,Purchase,1000.0,87.2994,2017-02-16


In [15]:
def get_inflation(date):

    if date <= pd.Timestamp("2018-11-01"):
        return 12.59

    elif date <= pd.Timestamp("2019-02-21"):
        return 10.93

    elif date <= pd.Timestamp("2019-08-07"):
        return 4.04

    elif date == pd.Timestamp("2019-08-08"):
        return 1.33

    else:
        return 0.0

In [16]:

def calculate_loss(buy, sell, qty):

    buy_price = buy["price"]
    sell_price = sell["Price per share"]

    buy_date = buy["date"]
    sell_date = sell["Trade Date"]

    inflation_buy = get_inflation(buy_date)
    inflation_sell = get_inflation(sell_date)

    if sell_date < pd.Timestamp("2018-11-02"):
        return 0

    elif sell_date <= pd.Timestamp("2019-08-07"):
        loss = min(
            inflation_buy - inflation_sell,
            buy_price - sell_price
        )

    elif sell_date <= pd.Timestamp("2019-11-05"):
        avg_price = 27.55
        loss = min(
            inflation_buy - inflation_sell,
            buy_price - avg_price,
            buy_price - sell_price
        )

    else:
        loss = min(
            inflation_buy,
            buy_price - 27.55
        )

    return max(loss, 0) * qty

In [17]:
from collections import deque
import pandas as pd

def compute_loss_for_fund(df_fund):

    inventory = deque()
    total_loss = 0

    for _, row in df_fund.iterrows():

        qty = row["Qty"]
        price = row["Price per share"]
        date = row["Trade Date"]

        # 🟢 BUY
        if qty > 0:
            inventory.append({
                "qty": qty,
                "price": price,
                "date": date
            })

        # 🔴 SELL
        elif qty < 0:
            sell_qty = abs(qty)

            while sell_qty > 0 and inventory:

                buy = inventory[0]
                matched_qty = min(sell_qty, buy["qty"])

                # 👉 Apply loss logic ONLY if after Nov 2, 2018
                if date >= pd.Timestamp("2018-11-02"):

                    loss = calculate_loss(buy, row, matched_qty)
                    total_loss += loss

                # Update inventory
                buy["qty"] -= matched_qty
                sell_qty -= matched_qty

                if buy["qty"] == 0:
                    inventory.popleft()

    return total_loss

In [18]:
results = []

for (client, fund), group in df.groupby(["Entity", "Fund Name"]):

    loss = compute_loss_for_fund(group)

    results.append({
        "Client": client,
        "Fund": fund,
        "Loss": loss
    })

result_df = pd.DataFrame(results)
print(result_df)

        Client     Fund      Loss
0   Client 1-1   Fund 3       0.0
1   Client 1-1   Fund 4  135500.0
2   Client 1-2  Fund 11       0.0
3   Client 1-2  Fund 15    4980.0
4   Client 1-2  Fund 16       0.0
..         ...      ...       ...
77    Client 2  Fund 89       0.0
78    Client 2  Fund 90       0.0
79    Client 2  Fund 93       0.0
80    Client 2  Fund 97       0.0
81    Client 2  Fund 98       0.0

[82 rows x 3 columns]


In [19]:
client_loss = result_df.groupby("Client")["Loss"].sum().reset_index()
print(client_loss)

       Client         Loss
0  Client 1-1  135500.0000
1  Client 1-2  213333.3500
2    Client 2   51251.8342


In [20]:
client_loss.to_csv("final_results.csv", index=False)

In [21]:
result_df.to_csv("fund_level_results.csv", index=False)